# 01 · ML Feature Engineering
Builds hand-crafted lexical/sentiment features, Empath features, and TF-IDF for **all three splits** (train / val / test).  
Saves feature matrices, fitted transformers, and labels so NB and SVM notebooks can reload without recomputing.

**Prerequisite:** run `00_data_preparation.ipynb` first.

In [1]:
import os, pickle, warnings
import numpy as np
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from empath import Empath
import joblib
warnings.filterwarnings('ignore')

INPUT_DIR = '../outputs/preprocessed/'
OUTPUT_DIR = '../outputs/featured/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load(name):
    with open(os.path.join(INPUT_DIR, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

X_ml_train = load('X_ml_train')
X_ml_val   = load('X_ml_val')
X_ml_test  = load('X_ml_test')
y_train    = load('y_train').to_numpy()
y_val      = load('y_val').to_numpy()
y_test     = load('y_test').to_numpy()

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')


Train: 27052 | Val: 5797 | Test: 5797


## 1 · Lexical + Sentiment Features (7 dimensions)
`word_count`, `avg_word_len`, `exclamation_count`, VADER `pos / neg / neu / compound`

In [2]:
analyzer = SentimentIntensityAnalyzer()

def lexical_sentiment_features(text: str) -> list:
    tokens          = text.split()
    word_count      = len(tokens)
    avg_word_len    = np.mean([len(t) for t in tokens]) if tokens else 0.0
    exclamation_cnt = text.count('!')
    vs = analyzer.polarity_scores(text)
    return [word_count, avg_word_len, exclamation_cnt,
            vs['pos'], vs['neg'], vs['neu'], vs['compound']]

def build_lex(series):
    return np.array([lexical_sentiment_features(t) for t in tqdm(series, desc='Lex+Sent')])

lex_train = build_lex(X_ml_train)
lex_val   = build_lex(X_ml_val)
lex_test  = build_lex(X_ml_test)
print('Lexical feature shape:', lex_train.shape)


Lex+Sent: 100%|██████████| 5797/5797 [00:14<00:00, 390.41it/s]

Lexical feature shape: (27052, 7)


## 2 · Empath Features (194 dimensions)

In [3]:
lexicon    = Empath()
EMPATH_CATS = list(lexicon.cats.keys())

def empath_features(text: str) -> list:
    scores = lexicon.analyze(text, categories=EMPATH_CATS, normalize=True)
    if scores is None:
        return [0.0] * len(EMPATH_CATS)
    return [scores.get(c, 0.0) or 0.0 for c in EMPATH_CATS]

def build_empath(series):
    return np.array([empath_features(t) for t in tqdm(series, desc='Empath')])

emp_train = build_empath(X_ml_train)
emp_val   = build_empath(X_ml_val)
emp_test  = build_empath(X_ml_test)
print('Empath feature shape:', emp_train.shape)


Empath: 100%|██████████| 5797/5797 [00:22<00:00, 252.53it/s]

Empath feature shape: (27052, 194)


## 3 · TF-IDF (unigrams + bigrams, max 50 k features)
Fitted **only on train** to prevent data leakage.

In [4]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, sublinear_tf=True)
tfidf_train = tfidf.fit_transform(tqdm(X_ml_train, desc='TF-IDF fit+transform'))
tfidf_val   = tfidf.transform(tqdm(X_ml_val,   desc='TF-IDF val'))
tfidf_test  = tfidf.transform(tqdm(X_ml_test,  desc='TF-IDF test'))
print('TF-IDF shape:', tfidf_train.shape)


TF-IDF test: 100%|██████████| 5797/5797 [00:01<00:00, 3683.00it/s]


TF-IDF shape: (27052, 50000)


## 4 · Combine & Scale Dense Features
Two combined matrices are produced:
- **`X_ml_feat_*`** — MaxAbsScaler (for SVM, supports negatives)
- **`X_nb_feat_*`** — MinMaxScaler → non-negative (required by ComplementNB)

In [5]:
dense_train = np.hstack([lex_train, emp_train])
dense_val   = np.hstack([lex_val,   emp_val])
dense_test  = np.hstack([lex_test,  emp_test])

# ── SVM variant: MaxAbsScaler ───────────────────────────────────────────────
mas = MaxAbsScaler()
dense_train_mas = mas.fit_transform(dense_train)
dense_val_mas   = mas.transform(dense_val)
dense_test_mas  = mas.transform(dense_test)

X_ml_feat_train = hstack([tfidf_train, csr_matrix(dense_train_mas)])
X_ml_feat_val   = hstack([tfidf_val,   csr_matrix(dense_val_mas)])
X_ml_feat_test  = hstack([tfidf_test,  csr_matrix(dense_test_mas)])

# ── NB variant: MinMaxScaler (non-negative) ─────────────────────────────────
mms = MinMaxScaler()
dense_train_mms = mms.fit_transform(dense_train)
dense_val_mms   = mms.transform(dense_val)
dense_test_mms  = mms.transform(dense_test)

X_nb_feat_train = hstack([tfidf_train, csr_matrix(dense_train_mms)])
X_nb_feat_val   = hstack([tfidf_val,   csr_matrix(dense_val_mms)])
X_nb_feat_test  = hstack([tfidf_test,  csr_matrix(dense_test_mms)])

print('SVM feature matrix (train):', X_ml_feat_train.shape)
print('NB  feature matrix (train):', X_nb_feat_train.shape)


SVM feature matrix (train): (27052, 50201)
NB  feature matrix (train): (27052, 50201)


## 5 · Save All Artefacts

In [ ]:
import scipy.sparse

# Feature matrices
artefacts = {
    'X_ml_feat_train' : X_ml_feat_train,
    'X_ml_feat_val'   : X_ml_feat_val,
    'X_ml_feat_test'  : X_ml_feat_test,
    'X_nb_feat_train' : X_nb_feat_train,
    'X_nb_feat_val'   : X_nb_feat_val,
    'X_nb_feat_test'  : X_nb_feat_test,
    'y_train_np'      : y_train,
    'y_val_np'        : y_val,
    'y_test_np'       : y_test,
}

for name, obj in artefacts.items():
    path = os.path.join(OUTPUT_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  Saved {name}')

# Fitted transformers
joblib.dump(tfidf, os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl'))
joblib.dump(mas,   os.path.join(OUTPUT_DIR, 'dense_scaler_mas.pkl'))
joblib.dump(mms,   os.path.join(OUTPUT_DIR, 'dense_scaler_mms.pkl'))
print('Saved tfidf_vectorizer, dense_scaler_mas, dense_scaler_mms')
print('\nFeature engineering complete.')


  Saved X_ml_feat_train
  Saved X_ml_feat_val
  Saved X_ml_feat_test
  Saved X_nb_feat_train
  Saved X_nb_feat_val
  Saved X_nb_feat_test
  Saved y_train_np
  Saved y_val_np
  Saved y_test_np
  Saved tfidf_vectorizer, dense_scaler_mas, dense_scaler_mms

Feature engineering complete.


: 